# Suite PAR — Partial evaluation

`Partials.OfAny(expression, variables)` evaluates what the variables it is given determine and leaves the rest a Basic
expression, the residual: a known subexpression becomes the literal of its value, Kleene's logic decides `and`, `or`
and `implies` where it can, and a let of a known value disappears once its body no longer refers to it. The residual
agrees with the expression wherever the expression evaluates without raising.

In [ ]:
import itertools

from mbse.Expressions import Domains as D, Evaluators as V, Expressions as E, Partials as PA, Translators as T
from mbse.Expressions.Dialects.Python import Expressions as P
from mbse.Expressions.Framework import Terms as F
from mbse.Schemas.Framework import JSON, Proxies, Schemas as S
from support import raises, text

Person = S.OfObject.Builder().ref().properties(text("name"), text("age", int), text("email")).create()
Proxies.register("Person", Person)
ann = Proxies.Builders.Person().name("Ann").age(30).email("ann@example.com").create()
bob = Proxies.Builders.Person().name("Bob").age(70).create()  # no email
this, x, y, xs = E.variable("this"), E.variable("x"), E.variable("y"), E.variable("xs")
I8 = D.OfInteger.Data(8)


def same(residual, expected):
    """Whether a residual is the expected expression."""
    return F.same(residual, E.OfAny.resolve(expected))

## PAR-01 · Known variables are evaluated and unknown ones stay; a known subexpression becomes the literal of its value

In [ ]:
assert same(PA.OfAny(x.add(E.literal(1).add(2))), x.add(3)) and same(PA.OfAny(x.add(1), {"x": 2}), 3)
assert same(PA.OfAny(E.literal(1).add(2).add(x)), E.literal(3).add(x)) and same(PA.OfAny(x), x)
assert same(PA.OfAny(x.add(E.literal(1, I8)), {"x": D.Value(I8, 2)}), E.literal(3, I8))  # a typed literal
assert same(PA.OfAny(this.age.ge(18), {"this": ann}), True) and same(PA.OfAny(this.has("email"), {"this": bob}), False)
# A value without a literal keeps its subexpression, which still refers to what it needs.
assert same(PA.OfAny(this, {"this": ann}), this) and same(PA.OfAny(xs, {"xs": [1, 2]}), xs)
assert same(PA.OfAny(xs.count(), {"xs": [1, 2]}), 2) and same(PA.OfAny(xs.count().add(y), {"xs": [1, 2]}), E.literal(2).add(y))
unknown = this.get("phone").ge(18)
assert same(PA.OfAny(unknown, {"this": ann}), unknown) and same(PA.OfAny(unknown.or_(y), {"this": ann}), unknown.or_(y))
assert same(PA.OfAny(lambda b: b.as_literal(1)), 1) and same(PA.OfAny(1), 1)  # any Spec

## PAR-02 · Kleene's logic decides and, or and implies with what is known, and drops a neutral operand

In [ ]:
rule = x.gt(1).and_(y.lt(2))
assert same(PA.OfAny(rule, {"x": 5}), y.lt(2)) and same(PA.OfAny(rule, {"x": 0}), False)
assert same(PA.OfAny(rule, {"y": 1}), x.gt(1)) and same(PA.OfAny(rule, {"y": 5}), False) and same(PA.OfAny(rule), rule)
either = x.gt(1).or_(y.lt(2))
assert same(PA.OfAny(either, {"x": 5}), True) and same(PA.OfAny(either, {"x": 0}), y.lt(2))
assert same(PA.OfAny(either, {"y": 1}), True) and same(PA.OfAny(either, {"y": 5}), x.gt(1))
then = x.implies(y)
assert same(PA.OfAny(then, {"x": False}), True) and same(PA.OfAny(then, {"y": True}), True)
assert same(PA.OfAny(then, {"x": True}), y) and same(PA.OfAny(then, {"y": False}), x.implies(False)) and same(PA.OfAny(then), then)
# An unknown value decides nothing: the operand stays.
assert same(PA.OfAny(rule, {"x": None}), rule) and same(PA.OfAny(x.not_(), {}), x.not_())
assert same(PA.OfAny(E.operation("xor", x, True)), E.operation("xor", x, True))  # an extension is only rebuilt

## PAR-03 · A let of a known value binds it in its body and stays while the body refers to it; names hide known variables

In [ ]:
a = E.variable("a")
rule = E.let_("a", x.add(1), a.mul(y))
assert same(PA.OfAny(rule, {"x": 2}), E.literal(3).mul(y)) and same(PA.OfAny(rule, {"y": 2}), E.let_("a", x.add(1), a.mul(2)))
assert same(PA.OfAny(rule), rule) and same(PA.OfAny(rule, {"x": 2, "y": 2}), 6)
o = E.variable("o")
assert same(PA.OfAny(E.let_("o", this, o.age.add(y)), {"this": ann}), E.literal(30).add(y))  # dropped: no longer needed
assert same(PA.OfAny(E.let_("o", this, o.add(y)), {"this": ann}), E.let_("o", this, o.add(y)))  # an object: kept
assert same(PA.OfAny(E.let_("x", y, x.add(1)), {"x": 5}), E.let_("x", y, x.add(1)))  # the let hides the known x
# Quantifiers: evaluated when known; otherwise their collection and body are reduced, the name hiding a known variable.
v = E.variable("v")
assert same(PA.OfAny(xs.all("v", v.gt(y)), {"xs": [1, 2], "y": 0}), True)
assert same(PA.OfAny(xs.all("v", v.gt(y)), {"y": 0}), xs.all("v", v.gt(0)))
assert same(PA.OfAny(xs.all("v", v.gt(y)), {"xs": [1, 2]}), xs.all("v", v.gt(y)))
assert same(PA.OfAny(xs.all("y", y.gt(0)), {"y": 5}), xs.all("y", y.gt(0)))

## PAR-04 · The residual agrees with evaluation in every scope that completes the known one, and stores and translates

In [ ]:
rule = this.age.ge(18).and_(x.lt(65).or_(this.has("email").not_())).implies(y.or_(x.eq(3)))
values = {"this": [ann, bob], "x": [3, 70], "y": [True, False, None]}
for size in range(4):
    for names in itertools.combinations(values, size):
        for known in itertools.product(*(values[name] for name in names)):
            given = dict(zip(names, known))
            residual = PA.OfAny(rule, given)
            others = [name for name in values if name not in given]
            for rest in itertools.product(*(values[name] for name in others)):
                scope = {**given, **dict(zip(others, rest))}
                assert V.OfAny(residual, scope) == V.OfAny(rule, scope), (given, scope)
residual = PA.OfAny(rule, {"this": bob})
assert same(residual, y.or_(x.eq(3)))  # Bob is an adult with no email: the condition holds
back = JSON.FromJSON(E.Builders).Reachable(E.OfOperation.Schema, JSON.ToJSON.Reachable(E.OfOperation.Schema, residual))
assert F.same(back, residual) and P.render(T.between(E.DIALECT, P.DIALECT).forward(residual)) == "y or x == 3"
# Evaluating a known subexpression raises as evaluation would; so does a cycle.
with raises(TypeError, match="add expects numbers of one domain, got int and str"):
    PA.OfAny(x.add(E.literal(1).add("a")))
cyclic = E.operation("add", 1, x).data
cyclic.arguments = (cyclic, x.data)
with raises(ValueError, match="the expression contains a cycle"):
    PA.OfAny(cyclic)
with raises(TypeError, match="not an expression: 'x'"):
    PA.REDUCER("x")